# SENTINEL Overseer — single-stage GRPO trainer (Colab fallback)

**Primary target:** Hugging Face Jobs (`scripts/launch_hf_job.sh`).
**This notebook:** identical pipeline, runnable cell-by-cell on Colab as a fallback.

Pipeline mirrors `training/grpo_hf_job.py`:

1. clone + install pinned deps
2. zero-shot Qwen3-1.7B baseline eval (the bar we have to clear)
3. apply LoRA adapter
4. SFT warmup on `training/sft_data/sft_warmup.jsonl` (1 epoch)
5. GRPO smoke (5 steps) — gates the long run
6. GRPO long run (400 steps, abort at 100/200 if reward stalls)
7. trained eval + `baseline_vs_trained.png` + `run_summary.json`
8. push LoRA to `Elliot89/sentinel-overseer-qwen3-1.7b` + git push artifacts

**Runtime guidance:** L4 / A100. Total wall clock ≈ 4 h on L4, ≈ 1.5 h on A100.

**Before running:**
- Add `HF_TOKEN` to Colab Secrets.
- Add `GITHUB_TOKEN` to Colab Secrets if you want auto-commit at the end.
- Set `SENTINEL_URL` env var if you want to point at a different env (default = the public Space).

## 0. Bootstrap — clone repo, install pinned deps

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = os.environ.get('GIT_REPO', 'https://github.com/MrEinsteinE/sentinel-openenv')
REPO_DIR = pathlib.Path(os.environ.get('SENTINEL_WORKDIR', '/content/sentinel-openenv'))
BRANCH = os.environ.get('GIT_BRANCH', 'main')

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Propagate so training.grpo_hf_job picks up the same path on import.
os.environ['SENTINEL_WORKDIR'] = str(REPO_DIR)
os.environ['SENTINEL_SKIP_BOOTSTRAP'] = '1'  # we already cloned manually above

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'training'))
print('repo at', REPO_DIR)

In [ ]:
%%capture
# Pinned dep set — must match training/grpo_hf_job.py PINS dict.
%pip install -q --upgrade pip
%pip install -q \
  'unsloth==2026.4.4' 'unsloth_zoo==2026.4.4' \
  'trl==0.21.0' 'transformers>=4.46.0,<4.47.0' \
  'peft>=0.13.0,<0.14.0' 'accelerate>=1.1.0,<1.3.0' \
  'bitsandbytes>=0.45.0' 'datasets>=2.18.0' \
  'huggingface_hub>=0.27.0' 'matplotlib>=3.8.0' 'numpy<2.0' \
  'fastapi>=0.104.0' 'uvicorn[standard]>=0.24.0' 'pydantic>=2.6.0' \
  'requests>=2.31.0' 'openai>=1.58.0'
# vLLM is large; conditional install. On non-CUDA runtimes this will fail loudly — that's fine, set SENTINEL_USE_VLLM=0 below.
%pip install -q 'vllm>=0.6.0,<0.7.0' || echo 'vllm install failed — set SENTINEL_USE_VLLM=0'

## 1. Configuration + auth

In [ ]:
import os

os.environ.setdefault('SENTINEL_URL', 'https://elliot89-sentinel.hf.space')
os.environ.setdefault('MODEL_NAME', 'unsloth/Qwen3-1.7B')
os.environ.setdefault('MODEL_REPO', 'Elliot89/sentinel-overseer-qwen3-1.7b')
os.environ.setdefault('STEP100_MIN_REWARD', '0.05')
os.environ.setdefault('STEP200_MIN_REWARD', '0.85')
os.environ.setdefault('SENTINEL_USE_VLLM', '1')

# Pull tokens from Colab Secrets when available.
try:
    from google.colab import userdata
    for k in ('HF_TOKEN', 'GITHUB_TOKEN'):
        try:
            os.environ[k] = userdata.get(k) or os.environ.get(k, '')
        except Exception:
            pass
except Exception:
    pass

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF login OK')
else:
    print('warn: HF_TOKEN not set — model push will be skipped')

## 2. Wake up the SENTINEL Space + smoke-test the env

In [ ]:
from training.grpo_hf_job import warmup_sentinel, build_tool_env_cls, SENTINEL_URL

warmup_sentinel(SENTINEL_URL)

ToolEnv = build_tool_env_cls(SENTINEL_URL)
_env = ToolEnv()
obs = _env.reset(task_id='action_screen', seed=1)
print('first observation:\n', obs[:400])

## 3. Load Qwen3-1.7B (4-bit QLoRA + vLLM colocate)

In [ ]:
from unsloth import FastLanguageModel

use_vllm = os.environ.get('SENTINEL_USE_VLLM', '1') == '1'
model, tokenizer = FastLanguageModel.from_pretrained(
    os.environ['MODEL_NAME'],
    max_seq_length=4096,
    load_in_4bit=True,
    fast_inference=use_vllm,
)
print('base model loaded; vllm=', use_vllm)

## 4. Zero-shot baseline (the F1 we must beat)

In [ ]:
from training.grpo_hf_job import _import_project, run_local_eval

project = _import_project()
baseline_summary = run_local_eval(model, tokenizer, 'qwen3_1_7b_zeroshot', project)
baseline_f1 = baseline_summary['per_task_f1']
print('zero-shot per-tier F1:', {k: v['f1'] for k, v in baseline_f1.items()})

## 5. Apply the LoRA adapter (rank 16) and run SFT warmup

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA applied')

In [ ]:
from training.grpo_hf_job import run_sft

run_sft(model, tokenizer, epochs=1, output_dir='outputs/sft_warmup_1ep')
print('SFT warmup complete')

## 6. GRPO smoke test (5 steps — gates the long run)

In [ ]:
from pathlib import Path
from training.grpo_hf_job import (
    TrackingCallback, _build_grpo_trainer, make_grpo_dataset,
    PLOTS_DIR, CKPT_DIR, SMOKE_STEPS,
)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

smoke_ds = make_grpo_dataset(n_samples=64)
smoke_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project['plot_loss'],
    plot_reward_fn=project['plot_reward'],
    is_smoke=True,
)
smoke_trainer = _build_grpo_trainer(
    model, tokenizer, smoke_ds, smoke_cb,
    output_dir='outputs/grpo_smoke', max_steps=SMOKE_STEPS, use_vllm=use_vllm,
)
smoke_trainer.train()
ok, msg = smoke_cb.smoke_pass()
print(msg)
assert ok, 'smoke failed — re-run cell 5b with epochs=2 then re-run this cell'

## 7. GRPO long run (400 steps, plots + checkpoints every 25)

If `abort_reason` is `'step100_resft'` after this cell, run cell 5b with `epochs=3` and re-run this cell. If `'step200_sft_only'`, skip directly to cell 8 — the SFT-only checkpoint is your final.

In [ ]:
from training.grpo_hf_job import GRPO_CONFIG

long_ds = make_grpo_dataset(n_samples=GRPO_CONFIG['max_steps'] * GRPO_CONFIG['gradient_accumulation_steps'])
long_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project['plot_loss'],
    plot_reward_fn=project['plot_reward'],
)
long_trainer = _build_grpo_trainer(
    model, tokenizer, long_ds, long_cb,
    output_dir='outputs/grpo_long', max_steps=GRPO_CONFIG['max_steps'], use_vllm=use_vllm,
)
long_trainer.train()
print('abort_reason =', long_cb.abort_reason)
print('best step =', long_cb.best_step, '@ reward', long_cb.best_reward)

## 8. Save best checkpoint + trained eval + comparison plot

In [ ]:
from training.grpo_hf_job import EVAL_DIR, _load_baselines

final_dir = CKPT_DIR / 'qwen3-1.7b-sentinel-best'
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print('saved adapter ->', final_dir)

trained_summary = run_local_eval(model, tokenizer, 'trained_qwen3_1_7b_grpo', project)
f1_per_tier = trained_summary['per_task_f1']

baselines = _load_baselines(EVAL_DIR)
baselines['qwen3_1_7b_zeroshot'] = baseline_f1
baselines['trained_qwen3_1_7b_grpo'] = f1_per_tier
project['plot_baseline_vs_trained'](
    baselines,
    trained_label='trained_qwen3_1_7b_grpo',
    out_path=str(PLOTS_DIR / 'baseline_vs_trained.png'),
    tier='action_screen',
)

from IPython.display import Image, display
for name in ('grpo_loss.png', 'grpo_reward.png', 'baseline_vs_trained.png'):
    p = PLOTS_DIR / name
    if p.exists():
        display(Image(filename=str(p)))

print(f'\nzero-shot action_screen F1: {baseline_f1["action_screen"]["f1"]:.3f}')
print(f'trained   action_screen F1: {f1_per_tier["action_screen"]["f1"]:.3f}')

## 9. Push LoRA to HF + git push artifacts

Push to model repo `Elliot89/sentinel-overseer-qwen3-1.7b` and git-commit `training/plots/`, `training/run_summary.json`, and the new eval JSONs to the GitHub repo.

In [ ]:
from training.grpo_hf_job import _write_summary, push_lora_to_hub, git_push_artifacts
import time

_write_summary(
    f1_per_tier=f1_per_tier,
    baseline_f1=baseline_f1,
    abort_path=long_cb.abort_reason,
    wall_clock_s=time.time(),
    best_step=long_cb.best_step,
)
push_lora_to_hub(final_dir)
git_push_artifacts(
    f'colab: training artifacts (action_screen F1 {f1_per_tier["action_screen"]["f1"]:.3f})'
)